# RAG Assignment — Baseline vs. Cross-Encoder Re-Ranking with Gemini

This notebook first extracts and cleans **text blocks only** from the configured Constitution of India PDF (no images or OCR), then compares two retrieval-augmented generation pipelines:

1. **Baseline:** Bi-Encoder → Qdrant Top-10 → first Top-3 → Gemini.
2. **Re-ranked:** Bi-Encoder → Qdrant Top-10 → Cross-Encoder rerank → reranked Top-3 → Gemini.

Both answer pipelines now reuse the **same exact Top-10 result for each query**. The intended experimental difference is which three passages are passed to Gemini.

**Models/services**
- Qdrant Cloud via REST API (used because the Python SDK timed out in this environment).
- Bi-Encoder: `mixedbread-ai/mxbai-embed-large-v1` (1024-dimensional vectors).
- Cross-Encoder: `mixedbread-ai/mxbai-rerank-base-v2`.
- Generator: `gemini-3.5-flash-lite` via Google's Gemini Developer API; free-tier access and quotas depend on the current account/model availability.

Keep `QDRANT_API_KEY` and `GEMINI_API_KEY` in `.env`; never print or commit them. This notebook contains no API keys.


## 0. Assignment requirements and experiment design

### Pipeline 1 — Baseline
```text
Query → Bi-Encoder → Qdrant Top-10 → first Top-3 → Gemini → Answer 1
```

### Pipeline 2 — Cross-Encoder Re-Ranking
```text
Query → Bi-Encoder → Qdrant Top-10 → Cross-Encoder → reranked Top-3 → Gemini → Answer 2
```

The two answer-generation paths use the same PDF, chunks, embedding model, initial Top-10 candidates, Gemini model, and prompt rules. The only planned difference is the ordering of the three context chunks supplied to Gemini.

The evaluation includes Recall@10, MRR@10, nDCG@10, Top-3 hit rate, rank shifts, latency, and side-by-side answer inspection. Six document-specific queries include positive cases, a reranker regression, and a retrieval miss to show that reranking is not guaranteed to help every query.

**Before running:** place the source PDF and a `.env` file in the notebook working directory. The `.env` file should contain:
```text
QDRANT_URL=https://your-cluster....qdrant.io
QDRANT_API_KEY=your_qdrant_api_key
GEMINI_API_KEY=your_gemini_api_key
```

Never share these keys or commit `.env`.


In [1]:
# Install dependencies. Run this cell once in the notebook environment.
%pip install -q \
    pymupdf \
    qdrant-client \
    sentence-transformers \
    transformers \
    accelerate \
    google-genai \
    python-dotenv \
    pandas \
    numpy \
    scikit-learn \
    tqdm \
    httpx


Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

- **Bi-Encoder:** `mixedbread-ai/mxbai-embed-large-v1` (local embeddings; 1024 dimensions).
- **Cross-Encoder:** `mixedbread-ai/mxbai-rerank-base-v2` (local reranking).
- **Generator:** `gemini-3.5-flash-lite` through the Gemini Developer API. Google lists a free tier for this model, subject to rate limits and account/model availability.


In [2]:
from pathlib import Path
import os
import re
import json
import time
import math
import statistics

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

# ---------------------------------------------------------
# Locate project directory
# ---------------------------------------------------------
# If the notebook is saved in your project directory, .env
# should be in the same directory.

PROJECT_DIR = Path.cwd()

# If you run the notebook from another directory, change this:
# PROJECT_DIR = Path("/path/to/your/project")

ENV_PATH = PROJECT_DIR / ".env"

if not ENV_PATH.exists():
    print(f"WARNING: .env was not found at: {ENV_PATH.resolve()}")
    print("Make sure the notebook and .env are in the same directory.")

load_dotenv(ENV_PATH)

# ---------------------------------------------------------
# User configuration
# ---------------------------------------------------------

# Override with RAG_PDF_PATH in .env if the PDF has a different name.
PDF_PATH = Path(os.getenv("RAG_PDF_PATH", "20240716890312078.pdf"))
if not PDF_PATH.is_absolute():
    PDF_PATH = PROJECT_DIR / PDF_PATH

# If the configured/default file is absent and exactly one PDF is in the
# working directory, use that PDF automatically. Otherwise fail clearly.
if not PDF_PATH.exists():
    pdf_candidates = sorted(PROJECT_DIR.glob("*.pdf"))
    if len(pdf_candidates) == 1:
        PDF_PATH = pdf_candidates[0]

COLLECTION_NAME = "rag_assignment"

EMBED_MODEL = "mixedbread-ai/mxbai-embed-large-v1"
RERANK_MODEL = "mixedbread-ai/mxbai-rerank-base-v2"
LLM_MODEL = "gemini-3.5-flash-lite"

TOP_K_RETRIEVAL = 10
TOP_K_CONTEXT = 3

CHUNK_SIZE_WORDS = 500
CHUNK_OVERLAP_WORDS = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set False to run retrieval/reranking without Gemini API calls.
USE_LLM = True

# Gemini API output-token limit for each answer.
# 256 was too small: Answer 2 for the test query was cut off mid-sentence.
MAX_NEW_TOKENS = 1024

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if USE_LLM and not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY is missing. Add it to .env before generating answers.")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Project directory:", PROJECT_DIR.resolve())
print("Environment file:", ENV_PATH.resolve())
print("PDF:", PDF_PATH)
print("Device:", DEVICE)
print("Embedding model:", EMBED_MODEL)
print("Reranker:", RERANK_MODEL)
print("LLM:", LLM_MODEL)


Project directory: /home/rdr/M26-LMA/assignments
Environment file: /home/rdr/M26-LMA/assignments/.env
PDF: /home/rdr/M26-LMA/assignments/20240716890312078.pdf
Device: cpu
Embedding model: mixedbread-ai/mxbai-embed-large-v1
Reranker: mixedbread-ai/mxbai-rerank-base-v2
LLM: gemini-3.5-flash-lite


## 2. Validate Qdrant Cloud credentials

The only Qdrant information required is:

```text
QDRANT_URL
QDRANT_API_KEY
```

These are read from `.env`.

The API key is never printed.


In [3]:

import os
from pathlib import Path
from dotenv import load_dotenv

# Use the current notebook directory
PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
print(".env file exists:", ENV_PATH.is_file())

# Load variables from the .env file
load_dotenv(dotenv_path=ENV_PATH, override=True)

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

if not QDRANT_URL or not QDRANT_API_KEY:
    raise ValueError(
        "QDRANT_URL or QDRANT_API_KEY is missing. "
        f"Check the .env file at: {ENV_PATH}"
    )

print("QDRANT_URL loaded:", bool(QDRANT_URL))
print("QDRANT_API_KEY loaded:", bool(QDRANT_API_KEY))


.env file exists: True
QDRANT_URL loaded: True
QDRANT_API_KEY loaded: True


## 3. Check Qdrant Cloud connectivity

The notebook uses Qdrant's REST API directly because the Python SDK timed out in this environment. The collection is managed in one place later, after the actual embedding dimension is known.

This cell only checks connectivity; it does not create or delete a collection.


In [4]:
%pip show qdrant-client httpx httpcore

Name: qdrant-client
Version: 1.19.1
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: /home/rdr/miniforge3/envs/RAG/lib/python3.11/site-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 
---
Name: httpx
Version: 0.28.1
Summary: The next generation HTTP client.
Home-page: https://github.com/encode/httpx
Author: 
Author-email: Tom Christie <tom@tomchristie.com>
License: BSD-3-Clause
Location: /home/rdr/miniforge3/envs/RAG/lib/python3.11/site-packages
Requires: anyio, certifi, httpcore, idna
Required-by: google-genai, huggingface_hub, qdrant-client
---
Name: httpcore
Version: 1.0.9
Summary: A minimal low-level HTTP client.
Home-page: https://www.encode.io/httpcore/
Author: 
Author-email: Tom Christie <tom@tomchristie.com>
License-Expression: BSD-3-Clause
Location: /home/rdr/miniforge3/envs/RAG/lib/

In [5]:
import os
import httpx
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path.cwd() / ".env", override=True)
url = os.getenv("QDRANT_URL", "").strip().rstrip("/")
api_key = os.getenv("QDRANT_API_KEY", "").strip()

if not url or not api_key:
    raise ValueError("QDRANT_URL or QDRANT_API_KEY is missing from .env.")

response = httpx.get(
    f"{url}/collections",
    headers={"api-key": api_key},
    timeout=httpx.Timeout(60.0, connect=15.0),
    trust_env=False,
)
response.raise_for_status()
print("Qdrant Cloud connectivity: OK")
print("Collections:", [x["name"] for x in response.json()["result"]["collections"]])


Qdrant Cloud connectivity: OK
Collections: ['rag_assignment']


In [6]:
# Collection creation is intentionally handled only in Section 7,
# after document_embeddings is available and its real dimension is known.
print("Collection setup is deferred until after document embeddings are created.")


Collection setup is deferred until after document embeddings are created.


In [7]:
# Quick credential/connectivity confirmation (does not create a collection).
response = httpx.get(
    f"{url}/collections",
    headers={"api-key": api_key},
    timeout=30,
    trust_env=False,
)
response.raise_for_status()
print("Qdrant REST API is reachable.")


Qdrant REST API is reachable.


## 4. Extract and clean text from the Constitution PDF

This step deliberately extracts **text blocks only** from the PDF. Images, scanned page pictures, logos, and other non-text blocks are not sent to the embedding model, Qdrant, the reranker, or Gemini. No OCR is run. If a page is image-only and has no selectable text layer, it is skipped and counted in the report rather than silently sending an image downstream.

Cleaning is conservative for legal text: Unicode/ligature normalization, removal of control/replacement characters, whitespace normalization, removal of standalone page-number lines and decorative separators, and removal of repeated headers/footers only when they recur at page edges. Article numbers, section references, punctuation, parentheses, and legal wording are retained.

The cleaned, page-labelled text is also written to `outputs/cleaned_pdf_text.txt` for inspection before chunking.


In [8]:
import fitz
import unicodedata
from collections import Counter

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH.resolve()}\n"
        "Update PDF_PATH in the configuration cell."
    )


def normalize_line(line):
    """Conservative text cleanup; preserve legal wording and punctuation."""
    line = unicodedata.normalize("NFKC", line)
    # Remove control characters and Unicode replacement characters, but keep tabs/newlines handled below.
    line = "".join(
        ch for ch in line
        if ch == "\t" or ch.isprintable()
    ).replace("\ufffd", "")
    line = re.sub(r"[ \t]+", " ", line).strip()
    return line


def header_footer_key(line):
    """Normalize a candidate edge line for detecting repeated page furniture."""
    return re.sub(r"[^a-z0-9]+", "", line.lower())


def extract_pages(pdf_path):
    """Extract text blocks only, then clean text while preserving source page numbers."""
    raw_pages = []
    doc = fitz.open(pdf_path)
    try:
        for page_idx, page in enumerate(doc):
            text_blocks = []
            # Block type 0 means text. Image blocks (typically type 1) are explicitly ignored.
            for block in page.get_text("blocks", sort=True):
                if len(block) >= 7 and block[6] == 0:
                    block_text = block[4]
                    if isinstance(block_text, str) and block_text.strip():
                        text_blocks.append(block_text)
            raw_text = "\n".join(text_blocks)
            lines = [normalize_line(line) for line in raw_text.splitlines()]
            lines = [line for line in lines if line]
            raw_pages.append({"page": page_idx + 1, "lines": lines})
    finally:
        doc.close()

    # Identify repeated lines only at the top/bottom edges of pages (common headers/footers).
    edge_counts = Counter()
    for item in raw_pages:
        lines = item["lines"]
        edge_lines = lines[:2] + (lines[-2:] if len(lines) > 2 else [])
        edge_counts.update(
            key for line in edge_lines
            if (key := header_footer_key(line)) and len(key) >= 4
        )
    min_repeat = max(3, math.ceil(len(raw_pages) * 0.20))
    repeated_edge_keys = {
        key for key, count in edge_counts.items() if count >= min_repeat
    }

    cleaned_pages = []
    stats = {
        "pdf_pages": len(raw_pages),
        "pages_with_text_before_cleaning": 0,
        "pages_with_text_after_cleaning": 0,
        "removed_repeated_header_footer_lines": 0,
        "removed_page_number_lines": 0,
        "removed_decorative_lines": 0,
    }

    for item in raw_pages:
        lines = item["lines"]
        if lines:
            stats["pages_with_text_before_cleaning"] += 1
        kept = []
        for line_idx, line in enumerate(lines):
            # Strip only page-number-only lines, not numbers embedded in legal clauses.
            if re.fullmatch(r"(?i)(?:page\s*)?\d{1,4}(?:\s*(?:of|/)\s*\d{1,4})?", line):
                stats["removed_page_number_lines"] += 1
                continue
            # Remove repeated header/footer text only when it appears at a page edge.
            is_edge = line_idx < 2 or line_idx >= max(0, len(lines) - 2)
            key = header_footer_key(line)
            if is_edge and key in repeated_edge_keys:
                stats["removed_repeated_header_footer_lines"] += 1
                continue
            if re.fullmatch(r"[-_=*~.·•\s]{3,}", line):
                stats["removed_decorative_lines"] += 1
                continue
            # Drop obvious extraction debris; do not apply this to short legal lines.
            if len(line) >= 12:
                alnum_count = sum(ch.isalnum() for ch in line)
                if alnum_count == 0:
                    continue
            kept.append(line)

        # Join words split across lines by PDF layout, but preserve normal hyphens.
        text = "\n".join(kept)
        text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
        # Keep paragraph boundaries where possible; collapse excessive blank space.
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            stats["pages_with_text_after_cleaning"] += 1
            cleaned_pages.append({"page": item["page"], "text": text})

    return cleaned_pages, stats


pages, pdf_cleaning_stats = extract_pages(PDF_PATH)

print("PDF cleaning report:")
for key, value in pdf_cleaning_stats.items():
    print(f"  {key}: {value}")
print(f"  pages_skipped_no_text: {pdf_cleaning_stats['pdf_pages'] - pdf_cleaning_stats['pages_with_text_after_cleaning']}")

# Save an auditable text-only extraction. This is not a new PDF; it is the cleaned text layer.
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
cleaned_text_path = OUTPUT_DIR / "cleaned_pdf_text.txt"
with cleaned_text_path.open("w", encoding="utf-8") as f:
    for item in pages:
        f.write(f"\n\n===== SOURCE PDF PAGE {item['page']} =====\n")
        f.write(item["text"])
        f.write("\n")

print("Cleaned text saved to:", cleaned_text_path.resolve())
print(f"Extracted {len(pages)} pages containing usable text.")

if pages:
    print("\nFirst cleaned page preview:")
    print(pages[0]["text"][:2000])


PDF cleaning report:
  pdf_pages: 402
  pages_with_text_before_cleaning: 402
  pages_with_text_after_cleaning: 402
  removed_repeated_header_footer_lines: 331
  removed_page_number_lines: 623
  removed_decorative_lines: 281
  pages_skipped_no_text: 0
Cleaned text saved to: /home/rdr/M26-LMA/assignments/outputs/cleaned_pdf_text.txt
Extracted 402 pages containing usable text.

First cleaned page preview:
£ÉÉ®iÉ BÉEÉ oÉÆÉÊ ́ÉvÉÉxÉ
[1
, 2024
]
THE CONSTITUTION OF INDIA
[As on 1st May, 2024]
GOVERNMENT OF INDIA
MINISTRY OF LAW AND JUSTICE
LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING


## 5. Chunk the document

Initial configuration:

- chunk size: **500 words**
- overlap: **100 words**

These values are starting points. For a policy/legal document, inspect the resulting chunks and make sure exceptions and restrictions are not separated from the rule they modify.


In [9]:
def chunk_words(text, chunk_size=500, overlap=100):
    words = text.split()

    if not words:
        return []

    step = max(1, chunk_size - overlap)
    chunks = []

    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]

        if not piece:
            break

        chunks.append(" ".join(piece))

        if start + chunk_size >= len(words):
            break

    return chunks


chunks = []
chunk_id = 0

for page in pages:
    page_chunks = chunk_words(
        page["text"],
        chunk_size=CHUNK_SIZE_WORDS,
        overlap=CHUNK_OVERLAP_WORDS
    )

    for local_idx, text in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"chunk_{chunk_id:05d}",
            "page": page["page"],
            "local_chunk": local_idx,
            "source": PDF_PATH.name,
            "text": text
        })
        chunk_id += 1

print("Total chunks:", len(chunks))

display(pd.DataFrame([
    {
        "chunk_id": c["chunk_id"],
        "page": c["page"],
        "words": len(c["text"].split()),
        "preview": c["text"][:250]
    }
    for c in chunks[:10]
]))


Total chunks: 406


,chunk_id,page,words,preview
0,chunk_00000,1,30,"£ÉÉ®iÉ BÉEÉ oÉÆÉÊ ́ÉvÉÉxÉ [1 , 2024 ] THE CONS..."
1,chunk_00001,2,137,PREFACE This is the sixth pocket size edition ...
2,chunk_00002,3,55,"LIST OF ABBREVIATIONS USED Art., arts. .........."
3,chunk_00003,4,150,CONTENTS PREAMBLE PART I THE UNION AND ITS TER...
4,chunk_00004,5,174,(ii) Contents ARTICLES PART III FUNDAMENTAL RI...
5,chunk_00005,6,195,(iii) Contents ARTICLES Cultural and Education...
6,chunk_00006,7,190,(iv) Contents ARTICLES 40. Organisation of vil...
7,chunk_00007,8,227,(v) Contents ARTICLES 55. Manner of election o...
8,chunk_00008,9,223,(vi) Contents ARTICLES Conduct of Government B...
9,chunk_00009,10,194,(vii) Contents ARTICLES 96. The Speaker or the...


## 6. Load the Bi-Encoder

The Bi-Encoder independently represents the document chunks and the query.

Document embeddings are generated once and stored in Qdrant.


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    EMBED_MODEL,
    device=DEVICE
)

document_texts = [c["text"] for c in chunks]

t0 = time.perf_counter()

document_embeddings = embedding_model.encode(
    document_texts,
    batch_size=16,
    show_progress_bar=True, 
    normalize_embeddings=True,
    convert_to_numpy=True
)

embedding_index_time = time.perf_counter() - t0

print("Embedding shape:", document_embeddings.shape)
print(f"Embedding/index preparation time: {embedding_index_time:.2f} sec")


/home/rdr/miniforge3/envs/RAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 26/26 [22:34<00:00, 52.10s/it]

Embedding shape: (406, 1024)
Embedding/index preparation time: 1354.59 sec


## 7. Create or reuse the Qdrant collection

The vector size is read from the actual embedding output. Cosine distance is used because the embeddings are normalized.

**Safety:** `RECREATE_COLLECTION = False` by default. If the collection already exists with the correct vector dimension, it is reused and point IDs are upserted. Set it to `True` only when you intentionally want to delete the existing `rag_assignment` collection and replace its contents. Deleting a collection permanently removes its existing points.


In [12]:
import httpx
import os

COLLECTION_NAME = "rag_assignment"
url = os.getenv("QDRANT_URL", "").strip().rstrip("/")
key = os.getenv("QDRANT_API_KEY", "").strip()
if not url or not key:
    raise ValueError("QDRANT_URL or QDRANT_API_KEY is missing. Run the credential cell and check .env.")
headers = {"api-key": key}
expected_vector_size = int(document_embeddings.shape[1])

# Safe default. Set True only if you explicitly want to delete and rebuild the collection.
RECREATE_COLLECTION = True

response = httpx.get(
    f"{url}/collections/{COLLECTION_NAME}",
    headers=headers,
    timeout=30,
    trust_env=False,
)

if response.status_code == 200 and RECREATE_COLLECTION:
    delete_response = httpx.delete(
        f"{url}/collections/{COLLECTION_NAME}",
        headers=headers,
        timeout=60,
        trust_env=False,
    )
    delete_response.raise_for_status()
    print(f"Deleted existing collection '{COLLECTION_NAME}' because RECREATE_COLLECTION=True.")
    response = httpx.get(
        f"{url}/collections/{COLLECTION_NAME}",
        headers=headers,
        timeout=30,
        trust_env=False,
    )

if response.status_code == 200:
    collection_info = response.json()["result"]
    vector_config = collection_info["config"]["params"]["vectors"]
    existing_vector_size = vector_config.get("size") if isinstance(vector_config, dict) else None
    if existing_vector_size != expected_vector_size:
        raise ValueError(
            f"Embedding dimension mismatch: collection has {existing_vector_size}, "
            f"but this model produces {expected_vector_size}. If you intend to replace "
            "the collection, set RECREATE_COLLECTION=True and rerun this cell."
        )
    existing_points = int(collection_info.get("points_count") or 0)
    if existing_points not in (0, len(chunks)):
        raise RuntimeError(
            f"The existing collection contains {existing_points} points, but this PDF "
            f"currently produces {len(chunks)} chunks. To avoid leaving stale points, "
            "review the collection and set RECREATE_COLLECTION=True only if you approve "
            "deleting and rebuilding it."
        )
    print(f"Using existing collection '{COLLECTION_NAME}' (vector size {existing_vector_size}, points {existing_points}).")

elif response.status_code == 404:
    create_response = httpx.put(
        f"{url}/collections/{COLLECTION_NAME}",
        headers=headers,
        json={
            "vectors": {
                "size": expected_vector_size,
                "distance": "Cosine",
            }
        },
        timeout=60,
        trust_env=False,
    )
    create_response.raise_for_status()
    print(f"Created collection '{COLLECTION_NAME}' with vector size {expected_vector_size}.")
else:
    response.raise_for_status()


Deleted existing collection 'rag_assignment' because RECREATE_COLLECTION=True.
Created collection 'rag_assignment' with vector size 1024.


## 8. Upload chunks and embeddings to Qdrant

Each point contains:

- vector;
- chunk ID;
- page;
- source PDF;
- original chunk text.

The original text is stored in Qdrant so retrieval returns everything needed for reranking and generation.


In [13]:
UPLOAD_BATCH_SIZE = 64
upload_url = f"{url}/collections/{COLLECTION_NAME}/points"

if len(chunks) != len(document_embeddings):
    raise ValueError(
        f"Chunk/embedding count mismatch: {len(chunks)} chunks vs "
        f"{len(document_embeddings)} embeddings."
    )

points = []

for i, (chunk, embedding) in enumerate(zip(chunks, document_embeddings)):
    # Support either text strings or metadata dictionaries
    payload = chunk if isinstance(chunk, dict) else {"text": str(chunk)}

    points.append({
        "id": i,
        "vector": np.asarray(embedding, dtype=np.float32).tolist(),
        "payload": payload,
    })

t0 = time.perf_counter()

with httpx.Client(
    timeout=httpx.Timeout(120.0, connect=15.0),
    trust_env=False,
) as client:
    for start in range(0, len(points), UPLOAD_BATCH_SIZE):
        batch = points[start:start + UPLOAD_BATCH_SIZE]

        response = client.put(
            upload_url,
            headers=headers,
            params={"wait": "true"},
            json={"points": batch},
        )

        if response.status_code not in (200, 201):
            raise RuntimeError(
                f"Upload failed for batch starting at {start}. "
                f"HTTP {response.status_code}: {response.text[:1000]}"
            )

        print(f"Uploaded {min(start + len(batch), len(points))}/{len(points)}")

upload_time = time.perf_counter() - t0

print(f"\nUploaded {len(points)} vectors.")
print(f"Upload time: {upload_time:.2f} sec")


Uploaded 64/406
Uploaded 128/406
Uploaded 192/406
Uploaded 256/406
Uploaded 320/406
Uploaded 384/406
Uploaded 406/406

Uploaded 406 vectors.
Upload time: 2.97 sec


In [14]:

response = httpx.get(
    f"{url}/collections/{COLLECTION_NAME}",
    headers=headers,
    timeout=30,
    trust_env=False,
)
response.raise_for_status()

info = response.json()["result"]
print("Collection:", COLLECTION_NAME)
print("Indexed points:", info.get("points_count"))
print("Vectors configured:", info["config"]["params"]["vectors"])


Collection: rag_assignment
Indexed points: 406
Vectors configured: {'size': 1024, 'distance': 'Cosine'}


## 9. Verify the Qdrant collection

This cell confirms that the expected number of chunks was indexed.


In [15]:
response = httpx.post(
    f"{url}/collections/{COLLECTION_NAME}/points/count",
    headers={"api-key": key},
    json={"exact": True},
    timeout=30,
    trust_env=False,
)

response.raise_for_status()
exact_count = response.json()["result"]["count"]

print("Exact point count:", exact_count)
print("Expected chunks:", len(chunks))

if exact_count != len(chunks):
    raise RuntimeError(
        f"Qdrant contains {exact_count} points, but this run has {len(chunks)} chunks. "
        "Rerun Section 7 with RECREATE_COLLECTION=True, then rerun upload and verification."
    )
print("Qdrant point count matches the current chunk count.")


Exact point count: 406
Expected chunks: 406
Qdrant point count matches the current chunk count.


## 10. Initial Top-10 retrieval

This is the shared retrieval stage for both experiments.

**Do not modify the Top-10 separately for the reranking pipeline.**

The same Top-10 returned here becomes the Cross-Encoder input.


In [16]:
import os
import time
import httpx
import pandas as pd

COLLECTION_NAME = "rag_assignment"
url = os.getenv("QDRANT_URL", "").strip().rstrip("/")
key = os.getenv("QDRANT_API_KEY", "").strip()
headers = {"api-key": key}

def retrieve_top_k(query, k=TOP_K_RETRIEVAL):
    t0 = time.perf_counter()
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).tolist()

    # Use the current Qdrant query endpoint first. Fall back to legacy search
    # only if this server does not support /points/query.
    with httpx.Client(
        timeout=httpx.Timeout(60.0, connect=15.0),
        trust_env=False,
    ) as client:
        response = client.post(
            f"{url}/collections/{COLLECTION_NAME}/points/query",
            headers=headers,
            json={
                "query": query_vector,
                "limit": int(k),
                "with_payload": True,
                "with_vector": False,
            },
        )
        if response.status_code in (404, 405):
            response = client.post(
                f"{url}/collections/{COLLECTION_NAME}/points/search",
                headers=headers,
                json={
                    "vector": query_vector,
                    "limit": int(k),
                    "with_payload": True,
                    "with_vector": False,
                },
            )

    response.raise_for_status()
    body = response.json()["result"]
    results = body.get("points", []) if isinstance(body, dict) else body

    latency = time.perf_counter() - t0
    retrieved = []
    for rank, hit in enumerate(results, start=1):
        item = dict(hit.get("payload") or {})
        item["initial_rank"] = rank
        item["retrieval_score"] = float(hit["score"])
        item.setdefault("chunk_id", hit["id"])
        retrieved.append(item)

    return retrieved, latency


TEST_QUERY = "What are the main restrictions described in the document?"
top10_test, retrieval_latency = retrieve_top_k(TEST_QUERY)

display(pd.DataFrame([
    {
        "rank": x["initial_rank"],
        "chunk_id": x.get("chunk_id"),
        "page": x.get("page"),
        "score": x["retrieval_score"],
        "preview": str(x.get("text", ""))[:220],
    }
    for x in top10_test
]))
print(f"Retrieved {len(top10_test)} chunks")
print(f"Retrieval latency: {retrieval_latency * 1000:.2f} ms")


,rank,chunk_id,page,score,preview
0,1,chunk_00005,6,0.612310,(iii) Contents ARTICLES Cultural and Education...
1,2,chunk_00014,15,0.611961,(xii) Contents ARTICLES 191. Disqualifications...
2,3,chunk_00004,5,0.602212,(ii) Contents ARTICLES PART III FUNDAMENTAL RI...
3,4,chunk_00017,18,0.600812,(xv) Contents ARTICLES 243A. Gram Sabha. 243B....
4,5,chunk_00046,47,0.599806,(Part III.—Fundamental Rights) 1[(1A) In makin...
5,6,chunk_00022,23,0.598147,"(xx) Contents ARTICLES CHAPTER III. PROPERTY, ..."
6,7,chunk_00021,22,0.592648,(xix) Contents ARTICLES 275. Grants from the U...
7,8,chunk_00172,172,0.584752,(Part IXB.—Co-operative Societies) 243ZO. Righ...
8,9,chunk_00048,49,0.583934,(Part III.—Fundamental Rights) (b) the express...
9,10,chunk_00037,38,0.582591,(Part III.—Fundamental Rights) (a) access to s...


Retrieved 10 chunks
Retrieval latency: 787.86 ms


# Pipeline 1 — Baseline Retrieval

```text
Query
  ↓
Bi-Encoder
  ↓
Qdrant
  ↓
Top 10
  ↓
Top 3
  ↓
LLM
  ↓
Answer 1
```

No reranking is performed.


In [17]:
def baseline_pipeline(query, top10=None, retrieval_latency=None):
    """Baseline uses the first three items from the shared Top-10 candidates."""
    if top10 is None:
        top10, measured_latency = retrieve_top_k(query, TOP_K_RETRIEVAL)
        if retrieval_latency is None:
            retrieval_latency = measured_latency
    if retrieval_latency is None:
        retrieval_latency = 0.0

    top3 = top10[:TOP_K_CONTEXT]
    return {
        "query": query,
        "top10": top10,
        "top3": top3,
        "retrieval_latency": retrieval_latency,
    }


# Pipeline 2 — Cross-Encoder Re-Ranking

```text
Query
  ↓
Bi-Encoder
  ↓
Qdrant
  ↓
Same Top 10
  ↓
Cross-Encoder
  ↓
Reranked Top 10
  ↓
Top 3
  ↓
LLM
  ↓
Answer 2
```


In [18]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    RERANK_MODEL,
    max_length=512,
    device=DEVICE
)

def reranking_pipeline(query, top10=None, retrieval_latency=None):
    """Rerank the supplied shared Top-10, or retrieve candidates if not supplied."""
    if top10 is None:
        top10, measured_latency = retrieve_top_k(query, TOP_K_RETRIEVAL)
        if retrieval_latency is None:
            retrieval_latency = measured_latency
    if retrieval_latency is None:
        retrieval_latency = 0.0

    pairs = [(query, item["text"]) for item in top10]
    rerank_start = time.perf_counter()
    scores = reranker.predict(pairs, show_progress_bar=False)
    rerank_latency = time.perf_counter() - rerank_start

    reranked = []
    for item, score in zip(top10, scores):
        row = dict(item)
        row["reranker_score"] = float(score)
        reranked.append(row)

    reranked.sort(key=lambda x: x["reranker_score"], reverse=True)
    for rank, item in enumerate(reranked, start=1):
        item["reranked_rank"] = rank

    return {
        "query": query,
        "top10": top10,
        "reranked": reranked,
        "top3": reranked[:TOP_K_CONTEXT],
        "retrieval_latency": retrieval_latency,
        "rerank_latency": rerank_latency,
        "total_retrieval_rerank_latency": retrieval_latency + rerank_latency,
    }


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1203.15it/s]


## 11. Rank-shift analysis

This is one of the most important outputs for the assignment.

Look for cases such as:

```text
Initial rank 8 → Reranked rank 1
```

where the promoted chunk contains the actual answer, exception, restriction, or condition.


In [19]:
def rank_shift_table(rerank_result):
    rows = []

    for item in rerank_result["reranked"]:
        rows.append({
            "chunk_id": item["chunk_id"],
            "page": item["page"],
            "initial_rank": item["initial_rank"],
            "reranked_rank": item["reranked_rank"],
            "rank_change": (
                item["initial_rank"]
                - item["reranked_rank"]
            ),
            "retrieval_score": item["retrieval_score"],
            "reranker_score": item["reranker_score"],
            "preview": item["text"][:250]
        })

    return (
        pd.DataFrame(rows)
        .sort_values("reranked_rank")
        .reset_index(drop=True)
    )


example_rerank = reranking_pipeline(TEST_QUERY)

display(rank_shift_table(example_rerank))


,chunk_id,page,initial_rank,reranked_rank,rank_change,retrieval_score,reranker_score,preview
0,chunk_00037,38,10,1,9,0.582591,0.990874,(Part III.—Fundamental Rights) (a) access to s...
1,chunk_00022,23,6,2,4,0.598147,0.989013,"(xx) Contents ARTICLES CHAPTER III. PROPERTY, ..."
2,chunk_00005,6,1,3,-2,0.612310,0.977023,(iii) Contents ARTICLES Cultural and Education...
3,chunk_00004,5,3,4,-1,0.602212,0.977023,(ii) Contents ARTICLES PART III FUNDAMENTAL RI...
4,chunk_00021,22,7,5,2,0.592648,0.977023,(xix) Contents ARTICLES 275. Grants from the U...
5,chunk_00046,47,5,6,-1,0.599806,0.972415,(Part III.—Fundamental Rights) 1[(1A) In makin...
6,chunk_00014,15,2,7,-5,0.611961,0.970688,(xii) Contents ARTICLES 191. Disqualifications...
7,chunk_00172,172,8,8,0,0.584752,0.955319,(Part IXB.—Co-operative Societies) 243ZO. Righ...
8,chunk_00048,49,9,9,0,0.583934,0.949669,(Part III.—Fundamental Rights) (b) the express...
9,chunk_00017,18,4,10,-6,0.600812,0.874077,(xv) Contents ARTICLES 243A. Gram Sabha. 243B....


## 12. Build the identical LLM prompt

The prompt is intentionally identical for both pipelines.

Only the evidence chunks change.

This ensures that differences between Answer 1 and Answer 2 can reasonably be attributed to retrieval/reranking rather than different generation instructions.


In [20]:
def format_context(context_chunks):
    blocks = []

    for i, chunk in enumerate(context_chunks, start=1):
        blocks.append(
            f"[Context {i} | Page {chunk['page']} | "
            f"{chunk['chunk_id']}]\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(blocks)


def build_prompt(query, context_chunks):
    context = format_context(context_chunks)

    return f"""You are a document question-answering assistant.

Answer the user's question using ONLY the supplied context.

Rules:
1. Do not invent facts that are not supported by the context.
2. Preserve important exceptions, restrictions, conditions, and negations.
3. If the supplied context is insufficient, say so explicitly.
4. Do not replace a specific exception with a more general rule.
5. Give a concise, direct answer.
6. Mention relevant page number(s) when possible.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""


## 13. Configure the Gemini generator

Both pipelines use the same Gemini model and prompt format. Only the selected evidence chunks differ: the baseline uses its initial Top-3, while the reranking pipeline uses the Cross-Encoder's Top-3.

Gemini is called through Google's API, so there is no local 7B model download. `gemini-3.5-flash-lite` is listed with a free API tier, subject to current rate limits and availability. See the [official pricing page](https://ai.google.dev/gemini-api/docs/pricing). Keep `GEMINI_API_KEY` in `.env` and never print or commit the key.


In [21]:
from google import genai
from google.genai import types
from google.genai import errors as genai_errors

llm_client = None

if USE_LLM:
    # Reload .env in case it was edited after the configuration cell ran.
    load_dotenv(ENV_PATH, override=True)
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

    if not GEMINI_API_KEY:
        raise ValueError(
            f"GEMINI_API_KEY is missing. Add it to {ENV_PATH.resolve()} "
            "and rerun this cell. Get a key from https://aistudio.google.com/apikey"
        )

    llm_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client configured for model:", LLM_MODEL)
else:
    print("Gemini generation skipped because USE_LLM=False.")


Gemini client configured for model: gemini-3.5-flash-lite


In [22]:
def generate_answer(prompt, max_new_tokens=MAX_NEW_TOKENS):
    """Generate an answer with Gemini using the supplied RAG prompt."""
    if not USE_LLM:
        return (
            "[LLM disabled. Set USE_LLM=True and configure GEMINI_API_KEY "
            "to generate Answer 1 / Answer 2.]"
        )

    if llm_client is None:
        raise RuntimeError(
            "Gemini client is not initialized. Run the Gemini setup cell first."
        )

    # The free tier has a low requests-per-minute quota, so retry on
    # 429 (rate limit) and 500/503 (temporary overload) with a growing wait.
    max_attempts = 5
    for attempt in range(1, max_attempts + 1):
        try:
            response = llm_client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    max_output_tokens=max_new_tokens,
                ),
            )
            break
        except genai_errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_attempts:
                wait = 20 * attempt
                print(f"Gemini returned {e.code}; retrying in {wait}s "
                      f"(attempt {attempt}/{max_attempts}).")
                time.sleep(wait)
                continue
            raise

    answer = (response.text or "").strip()
    if not answer:
        raise RuntimeError(
            "Gemini returned an empty response. Check API quota/model access "
            "and try again."
        )
    return answer


## 14. Run Answer 1 vs Answer 2

This function runs both retrieval pipelines and uses the same prompt structure and LLM.


In [23]:
def run_comparison(query):
    # Retrieve once so both systems receive the exact same Top-10 candidates.
    shared_top10, shared_retrieval_latency = retrieve_top_k(query, TOP_K_RETRIEVAL)

    baseline = baseline_pipeline(
        query,
        top10=shared_top10,
        retrieval_latency=shared_retrieval_latency,
    )
    reranked = reranking_pipeline(
        query,
        top10=shared_top10,
        retrieval_latency=shared_retrieval_latency,
    )

    prompt_1 = build_prompt(query, baseline["top3"])
    prompt_2 = build_prompt(query, reranked["top3"])

    t0 = time.perf_counter()
    answer_1 = generate_answer(prompt_1)
    llm_time_1 = time.perf_counter() - t0

    t0 = time.perf_counter()
    answer_2 = generate_answer(prompt_2)
    llm_time_2 = time.perf_counter() - t0

    return {
        "query": query,
        "shared_top10_ids": [x["chunk_id"] for x in shared_top10],
        "baseline": baseline,
        "reranked": reranked,
        "answer_1": answer_1,
        "answer_2": answer_2,
        "llm_time_1": llm_time_1,
        "llm_time_2": llm_time_2,
    }


result = run_comparison(TEST_QUERY)

print("=" * 70)
print("ANSWER 1 — BASELINE")
print("=" * 70)
print(result["answer_1"])

print("\n" + "=" * 70)
print("ANSWER 2 — RE-RANKED")
print("=" * 70)
print(result["answer_2"])


ANSWER 1 — BASELINE
Based on the supplied context, the document mentions the following restrictions:

* Restriction on rights conferred by Part III while martial law is in force in any area (Article 34, Page 6).
* Restriction on powers of Legislative Council as to Bills other than Money Bills (Article 197, Page 15).

ANSWER 2 — RE-RANKED
Based on the supplied context, the document describes the following restrictions (and powers to impose restrictions):

* **Trade, Commerce, and Intercourse:** Parliament has the power to impose restrictions on trade, commerce, and intercourse (Articles 302, Page 23), and there are restrictions on the legislative powers of the Union and States regarding trade and commerce (Article 303, Page 23), as well as restrictions on trade, commerce, and intercourse among States (Article 304, Page 23).
* **Fundamental Rights in Specific Contexts:** 
  * Parliament has the power to modify the rights conferred by Part III in their application to Forces, etc. (Article

## 15. Compare the evidence given to the LLM

The baseline should contain the initial Top-3.

The advanced pipeline should contain the Cross-Encoder's reranked Top-3.


In [24]:
def context_table(items, score_column=None):
    rows = []

    for rank, item in enumerate(items, start=1):
        row = {
            "rank": rank,
            "chunk_id": item["chunk_id"],
            "page": item["page"],
            "preview": item["text"][:300]
        }

        if score_column and score_column in item:
            row[score_column] = item[score_column]

        rows.append(row)

    return pd.DataFrame(rows)


print("BASELINE TOP-3")
display(
    context_table(
        result["baseline"]["top3"],
        "retrieval_score"
    )
)

print("\nRERANKED TOP-3")
display(
    context_table(
        result["reranked"]["top3"],
        "reranker_score"
    )
)


BASELINE TOP-3


,rank,chunk_id,page,preview,retrieval_score
0,1,chunk_00005,6,(iii) Contents ARTICLES Cultural and Education...,0.612310
1,2,chunk_00014,15,(xii) Contents ARTICLES 191. Disqualifications...,0.611961
2,3,chunk_00004,5,(ii) Contents ARTICLES PART III FUNDAMENTAL RI...,0.602212



RERANKED TOP-3


,rank,chunk_id,page,preview,reranker_score
0,1,chunk_00037,38,(Part III.—Fundamental Rights) (a) access to s...,0.990874
1,2,chunk_00022,23,"(xx) Contents ARTICLES CHAPTER III. PROPERTY, ...",0.989013
2,3,chunk_00005,6,(iii) Contents ARTICLES Cultural and Education...,0.977023


# 16. Failure-mode analysis

The assignment asks for at least two failure queries where baseline retrieval degrades but reranking restores relevance.

The queries below were chosen by probing the Constitution of India PDF. For every query, the relevant chunk was found by searching the extracted text for the clause that answers the question and reading the chunk (article and page listed below).

| ID | Category | What makes it hard for the Bi-Encoder | Evidence |
|---|---|---|---|
| Q1 | Keyword trap / distractor | "dissolved" matches many passages about dissolving *other* bodies (State Assemblies, District Councils); the answer is phrased as "shall not be subject to dissolution" | Art. 83(1), p. 70, `chunk_00069` |
| Q2 | Exception / corner case | General passages about Parliament sessions outrank the clause with the exception "except when both Houses of Parliament are in session" | Art. 123(1), p. 87, `chunk_00086` |
| Q3 | Negation / constraint | Asks whether *nominated* members are allowed to vote; the answer is only implied by the word "**elected** members" in the electoral college | Art. 54, p. 57, `chunk_00056` |
| Q4 | Buried evidence (rank 4–10) | The six-month condition for non-member Ministers sits in a chunk that starts mid-article | Art. 75(5), p. 66, `chunk_00065` |
| Q5 | Reranker regression (extra) | The Cross-Encoder prefers the near-identical High Court rule (Art. 220) over the Supreme Court rule | Art. 124(7), p. 89, `chunk_00088` |
| Q6 | Recall failure (extra) | Everyday wording ("tried twice for the same crime") vs. legal wording ("prosecuted and punished for the same offence more than once") | Art. 20(2), p. 42, `chunk_00041` |

Q5 and Q6 are included on purpose: they show the limits of reranking (a reranker can make things worse, and it cannot recover a chunk that is not in the initial Top-10).

Do not automatically assume that a higher Cross-Encoder score means the passage is correct.


In [25]:
# =========================================================
# Evaluation queries with manually verified evidence chunks
# =========================================================
#
# relevant_chunk_ids were verified by searching the chunk texts
# for the clause that answers each question and reading the
# chunk. "evidence" records the article and page for the report.
# =========================================================

EVALUATION_QUERIES = [
    {
        "id": "Q1",
        "category": "keyword_trap",
        "query": "Can the Council of States be dissolved?",
        "evidence": "Art. 83(1), p.70: the Council of States shall not be subject to dissolution; one-third of members retire every second year.",
        "relevant_chunk_ids": ["chunk_00069"]
    },
    {
        "id": "Q2",
        "category": "exception",
        "query": "Can the President promulgate an ordinance when both Houses of Parliament are in session?",
        "evidence": "Art. 123(1), p.87: ordinances may be promulgated at any time EXCEPT when both Houses of Parliament are in session.",
        "relevant_chunk_ids": ["chunk_00086"]
    },
    {
        "id": "Q3",
        "category": "negation_constraint",
        "query": "Are nominated members of Parliament allowed to vote in the election of the President?",
        "evidence": "Art. 54, p.57: electoral college consists of the ELECTED members of both Houses and of State Legislative Assemblies (so nominated members are excluded).",
        "relevant_chunk_ids": ["chunk_00056"]
    },
    {
        "id": "Q4",
        "category": "buried_evidence",
        "query": "Can a person who is not a member of either House of Parliament be appointed as a Minister?",
        "evidence": "Art. 75(5), p.66: a Minister who is not a member of either House for six consecutive months ceases to be a Minister.",
        "relevant_chunk_ids": ["chunk_00065"]
    },
    {
        "id": "Q5",
        "category": "reranker_regression",
        "query": "Can a retired Supreme Court judge plead or act in any court in India?",
        "evidence": "Art. 124(7), p.89: no person who has held office as a Judge of the Supreme Court shall plead or act in any court or before any authority in India.",
        "relevant_chunk_ids": ["chunk_00088"]
    },
    {
        "id": "Q6",
        "category": "recall_failure",
        "query": "Can a person be tried and punished twice for the same crime?",
        "evidence": "Art. 20(2), p.42: no person shall be prosecuted and punished for the same offence more than once.",
        "relevant_chunk_ids": ["chunk_00041"]
    }
]

# Sanity check: every labelled chunk id must exist in the current corpus.
known_chunk_ids = {c["chunk_id"] for c in chunks}
for q in EVALUATION_QUERIES:
    missing = set(q["relevant_chunk_ids"]) - known_chunk_ids
    if missing:
        raise ValueError(f"{q['id']}: unknown chunk ids {missing} (chunking changed?)")

with pd.option_context("display.max_colwidth", 120):
    display(pd.DataFrame(EVALUATION_QUERIES))


,id,category,query,evidence,relevant_chunk_ids
0,Q1,keyword_trap,Can the Council of States be dissolved?,"Art. 83(1), p.70: the Council of States shall not be subject to dissolution; one-third of members retire every secon...",[chunk_00069]
1,Q2,exception,Can the President promulgate an ordinance when both Houses of Parliament are in session?,"Art. 123(1), p.87: ordinances may be promulgated at any time EXCEPT when both Houses of Parliament are in session.",[chunk_00086]
2,Q3,negation_constraint,Are nominated members of Parliament allowed to vote in the election of the President?,"Art. 54, p.57: electoral college consists of the ELECTED members of both Houses and of State Legislative Assemblies ...",[chunk_00056]
3,Q4,buried_evidence,Can a person who is not a member of either House of Parliament be appointed as a Minister?,"Art. 75(5), p.66: a Minister who is not a member of either House for six consecutive months ceases to be a Minister.",[chunk_00065]
4,Q5,reranker_regression,Can a retired Supreme Court judge plead or act in any court in India?,"Art. 124(7), p.89: no person who has held office as a Judge of the Supreme Court shall plead or act in any court or ...",[chunk_00088]
5,Q6,recall_failure,Can a person be tried and punished twice for the same crime?,"Art. 20(2), p.42: no person shall be prosecuted and punished for the same offence more than once.",[chunk_00041]


## 17. Retrieval metrics

After manually verifying the relevant chunk IDs, calculate:

- Recall@10
- MRR@10
- nDCG@10
- Top-3 hit rate

These quantitative metrics complement the required qualitative analysis.


In [26]:
def reciprocal_rank(ranked_ids, relevant_ids):
    relevant_ids = set(relevant_ids)

    for rank, chunk_id in enumerate(
        ranked_ids,
        start=1
    ):
        if chunk_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def dcg_at_k(relevances, k):
    relevances = relevances[:k]

    return sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevances)
    )


def ndcg_at_k(ranked_ids, relevant_ids, k=10):
    relevant_ids = set(relevant_ids)

    relevances = [
        1 if chunk_id in relevant_ids else 0
        for chunk_id in ranked_ids[:k]
    ]

    ideal = [1] * min(len(relevant_ids), k)

    dcg = dcg_at_k(
        relevances,
        k
    )

    idcg = dcg_at_k(
        ideal,
        k
    )

    return dcg / idcg if idcg > 0 else 0.0


def evaluate_query_retrieval(query_info):
    query = query_info["query"]
    relevant_ids = query_info["relevant_chunk_ids"]

    shared_top10, shared_latency = retrieve_top_k(query, TOP_K_RETRIEVAL)
    baseline = baseline_pipeline(query, top10=shared_top10, retrieval_latency=shared_latency)
    reranked = reranking_pipeline(query, top10=shared_top10, retrieval_latency=shared_latency)

    baseline_ids = [
        x["chunk_id"]
        for x in baseline["top10"]
    ]

    reranked_ids = [
        x["chunk_id"]
        for x in reranked["reranked"]
    ]

    return {
        "id": query_info["id"],
        "category": query_info["category"],

        "baseline_recall@10": (
            len(set(baseline_ids[:10]) & set(relevant_ids)) / len(set(relevant_ids))
            if relevant_ids else 0.0
        ),

        "reranked_recall@10": (
            len(set(reranked_ids[:10]) & set(relevant_ids)) / len(set(relevant_ids))
            if relevant_ids else 0.0
        ),

        "baseline_mrr@10": reciprocal_rank(
            baseline_ids,
            relevant_ids
        ),

        "reranked_mrr@10": reciprocal_rank(
            reranked_ids,
            relevant_ids
        ),

        "baseline_ndcg@10": ndcg_at_k(
            baseline_ids,
            relevant_ids,
            10
        ),

        "reranked_ndcg@10": ndcg_at_k(
            reranked_ids,
            relevant_ids,
            10
        ),

        "baseline_top3_hit": int(
            any(
                x in relevant_ids
                for x in baseline_ids[:3]
            )
        ),

        "reranked_top3_hit": int(
            any(
                x in relevant_ids
                for x in reranked_ids[:3]
            )
        )
    }


In [27]:
valid_queries = [
    q
    for q in EVALUATION_QUERIES
    if q["relevant_chunk_ids"]
    and "REPLACE WITH" not in q["query"]
]

if valid_queries:
    metric_rows = [
        evaluate_query_retrieval(q)
        for q in valid_queries
    ]

    metrics_df = pd.DataFrame(metric_rows)

    display(metrics_df)

    print("\nMean metrics:")
    display(
        metrics_df[
            [
                "baseline_recall@10",
                "reranked_recall@10",
                "baseline_mrr@10",
                "reranked_mrr@10",
                "baseline_ndcg@10",
                "reranked_ndcg@10",
                "baseline_top3_hit",
                "reranked_top3_hit"
            ]
        ].mean().to_frame("mean")
    )
else:
    print(
        "Add real document-specific queries and "
        "verified relevant_chunk_ids first."
    )


,id,category,baseline_recall@10,reranked_recall@10,baseline_mrr@10,reranked_mrr@10,baseline_ndcg@10,reranked_ndcg@10,baseline_top3_hit,reranked_top3_hit
0,Q1,keyword_trap,1.0,1.0,0.125000,1.0,0.315465,1.000000,0,1
1,Q2,exception,1.0,1.0,0.200000,1.0,0.386853,1.000000,0,1
2,Q3,negation_constraint,1.0,1.0,0.250000,1.0,0.430677,1.000000,0,1
3,Q4,buried_evidence,1.0,1.0,0.166667,1.0,0.356207,1.000000,0,1
4,Q5,reranker_regression,1.0,1.0,0.500000,0.2,0.630930,0.386853,1,0
5,Q6,recall_failure,1.0,1.0,0.333333,1.0,0.500000,1.000000,1,1



Mean metrics:


,mean
baseline_recall@10,1.000000
reranked_recall@10,1.000000
baseline_mrr@10,0.262500
reranked_mrr@10,0.866667
baseline_ndcg@10,0.436689
reranked_ndcg@10,0.897809
baseline_top3_hit,0.333333
reranked_top3_hit,0.833333


## 18. Detailed rank-shift report

This produces the report-ready table for relevant passages.

The most valuable cases are passages that:

- begin outside the Top 3;
- are promoted into the Top 3;
- contain information that materially improves Answer 2.


In [28]:
def collect_rank_shift_records(evaluation_queries):
    rows = []

    for q in evaluation_queries:
        if not q["relevant_chunk_ids"]:
            continue

        result = reranking_pipeline(q["query"])

        relevant = set(
            q["relevant_chunk_ids"]
        )

        for item in result["reranked"]:
            if item["chunk_id"] in relevant:
                rows.append({
                    "query_id": q["id"],
                    "category": q["category"],
                    "query": q["query"],
                    "chunk_id": item["chunk_id"],
                    "page": item["page"],
                    "initial_rank": item["initial_rank"],
                    "reranked_rank": item["reranked_rank"],
                    "rank_change": (
                        item["initial_rank"]
                        - item["reranked_rank"]
                    ),
                    "in_baseline_top3": item["initial_rank"] <= TOP_K_CONTEXT,
                    "in_reranked_top3": item["reranked_rank"] <= TOP_K_CONTEXT,
                    "retrieval_score": item["retrieval_score"],
                    "reranker_score": item["reranker_score"]
                })

        # A relevant chunk that never reached the Top-10 cannot be reranked.
        found = {item["chunk_id"] for item in result["reranked"]}
        for chunk_id in relevant - found:
            rows.append({
                "query_id": q["id"],
                "category": q["category"],
                "query": q["query"],
                "chunk_id": chunk_id,
                "page": next(c["page"] for c in chunks if c["chunk_id"] == chunk_id),
                "initial_rank": None,
                "reranked_rank": None,
                "rank_change": None,
                "in_baseline_top3": False,
                "in_reranked_top3": False,
                "retrieval_score": None,
                "reranker_score": None
            })

    return pd.DataFrame(rows)


if valid_queries:
    rank_shift_df = collect_rank_shift_records(
        valid_queries
    )

    display(rank_shift_df)

    print("\nReport-ready summary:")
    for r in rank_shift_df.itertuples():
        if pd.isna(r.initial_rank):
            print(f"{r.query_id} ({r.category}): {r.chunk_id} (p.{r.page}) "
                  f"was NOT in the initial Top-10, so reranking could not help.")
        else:
            print(f"{r.query_id} ({r.category}): {r.chunk_id} (p.{r.page}) "
                  f"moved from rank #{int(r.initial_rank)} -> rank #{int(r.reranked_rank)}")
else:
    print("No labelled evaluation queries yet.")


,query_id,category,query,chunk_id,page,initial_rank,reranked_rank,rank_change,in_baseline_top3,in_reranked_top3,retrieval_score,reranker_score
0,Q1,keyword_trap,Can the Council of States be dissolved?,chunk_00069,70,8,1,7,False,True,0.554369,0.998968
1,Q2,exception,Can the President promulgate an ordinance when...,chunk_00086,87,5,1,4,False,True,0.701596,0.999869
2,Q3,negation_constraint,Are nominated members of Parliament allowed to...,chunk_00056,57,4,1,3,False,True,0.673487,0.997199
3,Q4,buried_evidence,Can a person who is not a member of either Hou...,chunk_00065,66,6,1,5,False,True,0.616948,0.999643
4,Q5,reranker_regression,Can a retired Supreme Court judge plead or act...,chunk_00088,89,2,5,-3,True,False,0.660320,0.995095
5,Q6,recall_failure,Can a person be tried and punished twice for t...,chunk_00041,42,3,1,2,True,True,0.503128,0.990292



Report-ready summary:
Q1 (keyword_trap): chunk_00069 (p.70) moved from rank #8 -> rank #1
Q2 (exception): chunk_00086 (p.87) moved from rank #5 -> rank #1
Q3 (negation_constraint): chunk_00056 (p.57) moved from rank #4 -> rank #1
Q4 (buried_evidence): chunk_00065 (p.66) moved from rank #6 -> rank #1
Q5 (reranker_regression): chunk_00088 (p.89) moved from rank #2 -> rank #5
Q6 (recall_failure): chunk_00041 (p.42) moved from rank #3 -> rank #1


## 19. Latency profiling

Model loading time is excluded.

For each query we measure:

- Bi-Encoder + Qdrant retrieval;
- Cross-Encoder reranking;
- total retrieval + reranking;
- reranking overhead.

Run several repetitions and report median latency.


In [29]:
def profile_retrieval_latency(
    queries,
    repetitions=3
):
    rows = []

    for query in queries:
        baseline_times = []
        rerank_times = []
        rerank_only_times = []

        for _ in range(repetitions):
            t0 = time.perf_counter()

            _ = retrieve_top_k(
                query,
                TOP_K_RETRIEVAL
            )

            baseline_times.append(
                time.perf_counter() - t0
            )

            rr = reranking_pipeline(query)

            rerank_times.append(
                rr["total_retrieval_rerank_latency"]
            )

            rerank_only_times.append(
                rr["rerank_latency"]
            )

        baseline_median = statistics.median(
            baseline_times
        )

        rerank_median = statistics.median(
            rerank_times
        )

        cross_encoder_median = statistics.median(
            rerank_only_times
        )

        rows.append({
            "query": query,
            "baseline_median_ms":
                baseline_median * 1000,

            "cross_encoder_median_ms":
                cross_encoder_median * 1000,

            "reranking_total_median_ms":
                rerank_median * 1000,

            "reranking_overhead_ms":
                (rerank_median - baseline_median) * 1000
        })

    return pd.DataFrame(rows)


profile_queries = [
    q["query"]
    for q in EVALUATION_QUERIES
    if "REPLACE WITH" not in q["query"]
]

if profile_queries:
    latency_df = profile_retrieval_latency(
        profile_queries
    )

    display(latency_df)

    print("\nMedian across all evaluation queries (ms):")
    latency_summary = latency_df.drop(columns="query").median().to_frame("median_ms")
    latency_summary.loc["overhead_vs_baseline_%"] = (
        100 * latency_summary.loc["reranking_overhead_ms", "median_ms"]
        / latency_summary.loc["baseline_median_ms", "median_ms"]
    )
    display(latency_summary.round(1))
else:
    print(
        "Add real evaluation queries before "
        "running latency profiling."
    )


,query,baseline_median_ms,cross_encoder_median_ms,reranking_total_median_ms,reranking_overhead_ms
0,Can the Council of States be dissolved?,757.052424,34407.279794,35131.183476,34374.131052
1,Can the President promulgate an ordinance when...,796.053944,34904.040114,35690.970991,34894.917047
2,Are nominated members of Parliament allowed to...,797.615894,34827.200147,35600.194767,34802.578873
3,Can a person who is not a member of either Hou...,814.569930,34557.605439,35348.545193,34533.975263
4,Can a retired Supreme Court judge plead or act...,806.362831,34587.778115,35322.689643,34516.326812
5,Can a person be tried and punished twice for t...,791.074846,34373.626787,35174.740447,34383.665601



Median across all evaluation queries (ms):


,median_ms
baseline_median_ms,796.8
cross_encoder_median_ms,34572.7
reranking_total_median_ms,35335.6
reranking_overhead_ms,34525.2
overhead_vs_baseline_%,4332.8


## 20. Optional end-to-end answer comparison table

For the final report, record the baseline and reranked answers for your selected failure queries.

The qualitative evaluation should answer:

- Did Answer 1 omit a critical clause?
- Did Answer 1 confuse a general rule with an exception?
- Did Answer 2 recover the relevant condition?
- Did reranking introduce any new error?


In [30]:
def run_answer_evaluation(query_info):
    comparison = run_comparison(
        query_info["query"]
    )

    return {
        "id": query_info["id"],
        "category": query_info["category"],
        "query": query_info["query"],
        "baseline_top3_ids":
            [x["chunk_id"] for x in comparison["baseline"]["top3"]],
        "reranked_top3_ids":
            [x["chunk_id"] for x in comparison["reranked"]["top3"]],
        "relevant_chunk_ids":
            query_info["relevant_chunk_ids"],
        "answer_1_baseline":
            comparison["answer_1"],
        "answer_2_reranked":
            comparison["answer_2"],
        "llm_time_1_ms":
            comparison["llm_time_1"] * 1000,
        "llm_time_2_ms":
            comparison["llm_time_2"] * 1000,
        "baseline_retrieval_ms":
            comparison["baseline"]["retrieval_latency"] * 1000,
        "rerank_total_ms":
            comparison["reranked"][
                "total_retrieval_rerank_latency"
            ] * 1000,
        "rerank_overhead_ms":
            comparison["reranked"]["rerank_latency"] * 1000
    }


if valid_queries and USE_LLM:
    answer_rows = [
        run_answer_evaluation(q)
        for q in valid_queries
    ]

    answers_df = pd.DataFrame(answer_rows)

    display(answers_df)
else:
    print(
        "Enable USE_LLM and add labelled evaluation "
        "queries to generate the answer comparison table."
    )


,id,category,query,baseline_top3_ids,reranked_top3_ids,relevant_chunk_ids,answer_1_baseline,answer_2_reranked,llm_time_1_ms,llm_time_2_ms,baseline_retrieval_ms,rerank_total_ms,rerank_overhead_ms
0,Q1,keyword_trap,Can the Council of States be dissolved?,"[chunk_00129, chunk_00111, chunk_00071]","[chunk_00069, chunk_00110, chunk_00072]",[chunk_00069],The supplied context does not contain informat...,"No, the Council of States is not subject to di...",1277.781703,1092.886014,741.471582,35166.421266,34424.949684
1,Q2,exception,Can the President promulgate an ordinance when...,"[chunk_00070, chunk_00079, chunk_00242]","[chunk_00086, chunk_00070, chunk_00240]",[chunk_00086],The supplied context is insufficient to answer...,No. The President may promulgate an Ordinance ...,1154.507142,1153.402833,810.946852,35302.668559,34491.721707
2,Q3,negation_constraint,Are nominated members of Parliament allowed to...,"[chunk_00058, chunk_00077, chunk_00062]","[chunk_00056, chunk_00057, chunk_00070]",[chunk_00056],"Based on the supplied context, there is no men...","Based on the supplied context, the electoral c...",1470.229960,1173.304669,806.459297,35267.678548,34461.219251
3,Q4,buried_evidence,Can a person who is not a member of either Hou...,"[chunk_00064, chunk_00075, chunk_00380]","[chunk_00065, chunk_00064, chunk_00076]",[chunk_00065],"Based on the supplied context, there is no men...","Yes, but a Minister who for any period of six ...",1498.355186,1092.241717,811.778648,35128.817737,34317.039089
4,Q5,reranker_regression,Can a retired Supreme Court judge plead or act...,"[chunk_00134, chunk_00088, chunk_00133]","[chunk_00132, chunk_00092, chunk_00207]",[chunk_00088],No person who has held office as a Judge of th...,"Based on the supplied context, there is no men...",1315.886866,1198.237695,813.648274,35228.369816,34414.721542
5,Q6,recall_failure,Can a person be tried and punished twice for t...,"[chunk_00324, chunk_00042, chunk_00041]","[chunk_00041, chunk_00324, chunk_00042]",[chunk_00041],"No, no person shall be prosecuted and punished...","No. Under Article 20(2), no person shall be pr...",1217.877136,1164.003141,761.097684,35136.808275,34375.710591


In [31]:
# Print Answer 1 vs Answer 2 in full for the qualitative comparison
# (the DataFrame above truncates long text).
if "answers_df" in globals():
    for r in answers_df.itertuples():
        relevant = set(r.relevant_chunk_ids)
        print("=" * 90)
        print(f"{r.id} [{r.category}] {r.query}")
        print(f"Relevant chunk: {sorted(relevant)}")
        print(f"Baseline Top-3: {r.baseline_top3_ids}  "
              f"-> evidence included: {bool(relevant & set(r.baseline_top3_ids))}")
        print(f"Reranked Top-3: {r.reranked_top3_ids}  "
              f"-> evidence included: {bool(relevant & set(r.reranked_top3_ids))}")
        print("-" * 90)
        print("ANSWER 1 (baseline):\n" + r.answer_1_baseline)
        print("-" * 90)
        print("ANSWER 2 (reranked):\n" + r.answer_2_reranked)
        print()


Q1 [keyword_trap] Can the Council of States be dissolved?
Relevant chunk: ['chunk_00069']
Baseline Top-3: ['chunk_00129', 'chunk_00111', 'chunk_00071']  -> evidence included: False
Reranked Top-3: ['chunk_00069', 'chunk_00110', 'chunk_00072']  -> evidence included: True
------------------------------------------------------------------------------------------
ANSWER 1 (baseline):
The supplied context does not contain information on whether the Council of States can be dissolved (Context 3 discusses officers of the Council of States, while Contexts 1 and 2 relate to State Legislatures and Legislative Assemblies).
------------------------------------------------------------------------------------------
ANSWER 2 (reranked):
No, the Council of States is not subject to dissolution (Page 70, Context 1).

Q2 [exception] Can the President promulgate an ordinance when both Houses of Parliament are in session?
Relevant chunk: ['chunk_00086']
Baseline Top-3: ['chunk_00070', 'chunk_00079', 'chunk

## 21. Save experiment outputs

Outputs are stored locally in `outputs/`.

Generated files include:

- `retrieval_metrics.csv`
- `rank_shifts.csv`
- `latency_results.csv`
- `answer_comparisons.csv`
- `experiment_config.json`

**Do not commit `.env` to GitHub.**


In [32]:
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

if "metrics_df" in globals():
    metrics_df.to_csv(
        OUTPUT_DIR / "retrieval_metrics.csv",
        index=False
    )

if "rank_shift_df" in globals():
    rank_shift_df.to_csv(
        OUTPUT_DIR / "rank_shifts.csv",
        index=False
    )

if "latency_df" in globals():
    latency_df.to_csv(
        OUTPUT_DIR / "latency_results.csv",
        index=False
    )

if "answers_df" in globals():
    answers_df.to_csv(
        OUTPUT_DIR / "answer_comparisons.csv",
        index=False
    )

config = {
    "embedding_model": EMBED_MODEL,
    "reranker_model": RERANK_MODEL,
    "llm_provider": "Google Gemini Developer API",
    "llm_model": LLM_MODEL,
    "llm_free_tier_note": "Free-tier availability and quotas depend on Google current API terms.",
    "vector_database": "Qdrant Cloud",
    "collection_name": COLLECTION_NAME,
    "top_k_retrieval": TOP_K_RETRIEVAL,
    "top_k_context": TOP_K_CONTEXT,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "device": DEVICE,
    # The Qdrant cluster URL is deliberately not saved, so this file is safe to commit.
    "pdf": PDF_PATH.name,
    "pdf_cleaning": pdf_cleaning_stats,
    "cleaned_text_output": str((OUTPUT_DIR / "cleaned_pdf_text.txt").relative_to(PROJECT_DIR)),
    "max_new_tokens": MAX_NEW_TOKENS,
    "evaluation_queries": EVALUATION_QUERIES
}

# Do NOT save the API key.
with open(
    OUTPUT_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

print("Outputs saved to:")
print(OUTPUT_DIR.resolve())


Outputs saved to:
/home/rdr/M26-LMA/assignments/outputs


## 22. How to interpret the results and final checklist

- **Recall@10 should be identical** for baseline and reranked results when both rank the same initial Top-10 candidates. Reranking changes order; it cannot retrieve a candidate that was absent from the Top-10.
- **MRR@10 and nDCG@10** measure where the relevant evidence appears in the ranking. Higher is better.
- **Top-3 hit rate** is especially relevant to generation here because only the Top-3 chunks are passed to Gemini.
- A higher Cross-Encoder score is a ranking score, not proof that the passage is legally or factually correct. Inspect the chunk text and page.
- Q5 is an intentional regression example; Q6 is an intentional retrieval miss. These prevent the report from overstating reranking as universally successful.
- CPU latency can be high because the Cross-Encoder evaluates each query–chunk pair locally. Report actual measured latency and hardware; do not claim the reranker is faster based only on relevance metrics.

### Before submission
1. Run every cell from top to bottom in the intended environment.
2. Verify the PDF path and inspect `outputs/cleaned_pdf_text.txt` plus the extracted text/chunks; confirm image-only pages are skipped rather than OCRed.
3. Confirm the Qdrant exact point count equals the current number of chunks.
4. Inspect all six query results and both generated answers; correct any evidence labels that no longer match after changing the PDF or chunking.
5. Save the CSV outputs and `experiment_config.json` from the same final run.
6. Keep `.env` private and do not include API keys in the submission.

**Reproducibility note:** the six labelled chunk IDs are tied to the current PDF and chunking settings (500 words, 100-word overlap). If either changes, re-verify the relevant chunk IDs before interpreting the metrics.
